In [1]:
# LIBRERIE

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("cmi_internet.csv")
legenda = pd.read_csv("data_dictionary.csv")

# Stagioni enceoded winter=0, spring=1, summer=2, autumn=3

df['Fitness_Endurance-Tempo_totale'] = df['Fitness_Endurance-Time_Sec'] + \
    (df['Fitness_Endurance-Time_Mins']*60)

In [ ]:
# Colonne just to check
col_categoriche = df.select_dtypes(include=["object"]).columns
col_numeriche = df.select_dtypes(
    include=["int64", "float64"]).columns
col_quantitative = df.select_dtypes(
    include=["int64", "float64"]).columns.drop("id")

col_da_escludere = [
    "FGC-FGC_CU_Zone", "FGC-FGC_GSND_Zone", "FGC-FGC_GSD_Zone",
    "FGC-FGC_PU_Zone", "FGC-FGC_SRL_Zone", "FGC-FGC_SRR_Zone",
    "FGC-FGC_TL_Zone"
]

col_non_bin = col_quantitative.difference(col_da_escludere)

categoriche = df[col_categoriche]
stagioni = {"Winter": 0, "Spring": 1, "Summer": 2, "Fall": 3}
tutto_numerico = df.copy()
tutto_numerico[col_categoriche] = tutto_numerico[col_categoriche].replace(
    stagioni)
correlazione = tutto_numerico.corr()

In [ ]:
sns.set_theme(style="white")


def plot_professional_matrix(corr_matrix, title="Correlation Matrix"):
    # Creazione di una singola figura
    fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

    # Maschera per nascondere la parte superiore (triangolo)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    # Palette divergente professionale (es. Rosso-Blu)
    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    # Generazione della heatmap
    sns.heatmap(corr_matrix,
                mask=mask,
                cmap=cmap,
                vmax=1, vmin=-1, center=0,
                square=True,
                linewidths=.5,
                cbar_kws={"shrink": .5},
                annot=False,             # Impostare a True per mostrare i numeri
                fmt=".2f",               # Formato arrotondamento decimali
                ax=ax)

    # Impostazione del titolo
    ax.set_title(title, fontsize=16, pad=20)

    plt.show()


plot_professional_matrix(correlazione)

In [ ]:
# Fuori Scala

# CGAS-CGAS_Score
df["CGAS-CGAS_Score"] = df["CGAS-CGAS_Score"].mask(lambda x: x > 100, np.nan)

# Physical BMI
df["FGC-FGC_GSD"].value_counts()

In [ ]:
# Rimuovere 18+
# 0 = nan da lista
# paq Da arrotondare

df = df[df["Basic_Demos-Age"] <= 18]

df['PAQ_A-PAQ_A_Total'] = df['PAQ_A-PAQ_A_Total'].clip(lower=1)
df['PAQ_C-PAQ_C_Total'] = df['PAQ_C-PAQ_C_Total'].clip(lower=1)

colonne_zeri = ['Physical-BMI', 'Physical-Weight',
                'Physical-Diastolic_BP', 'Physical-Systolic_BP']
df[colonne_zeri] = df[colonne_zeri].mask(lambda x: x == 0, np.nan)


# train
train_set, test_set = train_test_split(
    df, test_size=0.3, stratify=df["sii"], random_state=222)

train_set = train_set.copy()
test_set = test_set.copy()

In [ ]:
# imputazione
# Ci ho dormito sopra, usare Knn (nonostante lo abbia detto l'altro Riccardo) è sbagliato su più punti, va rifatta l'imputazione con Random Forest o simili
# Intanto non si è fatto feature scaling prima del calcolo delle distanze
# Poi, se imputiamo dati mancanti di un'unità kinda vicina ad outliers ce la possibilità che si vada a creare un cluster, ed i metodi a densità non li identificano come tali
# Infine ce anche un discorso di facile propagazione di valori outliers, che potrebbero influenzare u po' troppo l'intero dataset

imputer = KNNImputer(n_neighbors=76)  # np.sqrt(5823)
train_imputed = pd.DataFrame(
    imputer.fit_transform(train_set[col_non_bin]),
    columns=col_non_bin,
    index=train_set.index)

test_imputed = pd.DataFrame(
    imputer.transform(test_set[col_non_bin]),
    columns=col_non_bin,
    index=test_set.index)

In [ ]:
train_imputed["Fitness_Endurance-Max_Stage"].value_counts()

In [ ]:
train_set["Fitness_Endurance-Max_Stage"].value_counts()

In [ ]:
'''from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

rf_estimator = RandomForestRegressor(random_state=222)
imputer_rf = IterativeImputer(estimator=rf_estimator, random_state=222, max_iter=5)

train_imputed_rf = pd.DataFrame(
    imputer_rf.fit_transform(train_set[col_non_bin]), 
    columns=col_non_bin, 
    index=train_set.index
)

test_imputed_rf = pd.DataFrame(
    imputer_rf.transform(test_set[col_non_bin]), 
    columns=col_non_bin, 
    index=test_set.index
)'''

In [ ]:
# Salvataggio dati in CSV

train_imputed.to_csv("train_imputed.csv", index=True)
test_imputed.to_csv("test_imputed.csv", index=True)